# 05 – CARMA MLE Fitting

Fits CARMA(p, q) models for both the temperature and log-price residual series  
by **maximum likelihood** using the Kalman filter log-likelihood.

### Key fixes vs. original
- **LI-1 fixed**: parameters are *optimised*, not hard-coded from ARMA moments.
- **LI-3 fixed**: time axis in **years** (`t = index / 8760`); CARMA parameters  
  are therefore in year⁻¹ units, consistent with the paper's pricing formulas.
- **LI-7 fixed**: initial covariance P₀ set to the **stationary** Lyapunov solution.
- **NCI-3 fixed**: Joseph-form covariance update in the Kalman filter.
- **NCI-1 fixed**: discrete transition F, Q_d computed once (equal spacing).

Inputs:  `data/deseasonalised/*_train.csv`, `results/carma_init.json`  
Outputs: `results/carma_temp_mle.json`, `results/carma_price_mle.json`,  
         `x_hat_temp.npy`, `x_hat_price.npy`

In [ ]:
import sys, os, json
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.stats.diagnostic import acorr_ljungbox

from config import (
    TEMP_RESID_TRAIN_CSV, PRICE_RESID_TRAIN_CSV,
    RES_DIR, FIG_DIR, HOURS_PER_YEAR, DT_YEARS,
)
from carma_utils import (
    build_companion, kalman_filter, fit_carma_mle,
    save_params, load_params,
)

# ── Load training residuals ───────────────────────────────────────────────────
temp_df  = pd.read_csv(TEMP_RESID_TRAIN_CSV,  index_col=0, parse_dates=True)
price_df = pd.read_csv(PRICE_RESID_TRAIN_CSV, index_col=0, parse_dates=True)

y_temp  = temp_df.iloc[:, 0].dropna().to_numpy(dtype=float)
y_price = price_df.iloc[:, 0].dropna().to_numpy(dtype=float)

# ── Time axis in YEARS (LI-3 fix) ────────────────────────────────────────────
t_temp  = np.arange(len(y_temp),  dtype=float) * DT_YEARS
t_price = np.arange(len(y_price), dtype=float) * DT_YEARS

print(f'Temperature  : {len(y_temp):,} obs   dt={DT_YEARS:.6f} yr')
print(f'Log-price    : {len(y_price):,} obs   dt={DT_YEARS:.6f} yr')

# ── Load starting points from notebook 04 ────────────────────────────────────
with open(RES_DIR / 'carma_init.json') as f:
    init_data = json.load(f)

theta0_temp  = [np.array(th) for th in init_data['temperature']['theta0_list']]
theta0_price = [np.array(th) for th in init_data['logprice']['theta0_list']]

p_t, q_t = init_data['temperature']['p'], init_data['temperature']['q']
p_p, q_p = init_data['logprice']['p'],    init_data['logprice']['q']

print(f'\nCARMA orders: temperature=({p_t},{q_t})  log-price=({p_p},{q_p})')
print(f'Starting points: {len(theta0_temp)} (temp)  {len(theta0_price)} (price)')

## 1.  Temperature CARMA MLE

This cell may take a few minutes (multi-start L-BFGS-B over ~52,000 obs).

In [ ]:
TEMP_MLE_PATH = RES_DIR / 'carma_temp_mle.json'

# Shortcut: reload if already computed
if TEMP_MLE_PATH.exists():
    print('Loading cached temperature MLE result …')
    params_temp = load_params(TEMP_MLE_PATH)
else:
    print(f'Fitting CARMA({p_t},{q_t}) to temperature (multi-start MLE) …')
    _, params_temp = fit_carma_mle(
        t_years     = t_temp,
        y           = y_temp,
        p           = p_t,
        q           = q_t,
        theta0_list = theta0_temp,
        verbose     = True,
    )
    save_params(params_temp, TEMP_MLE_PATH)

print(f'\nTemperature CARMA({p_t},{q_t}) MLE result:')
print(f'  a_coeffs  : {np.round(params_temp["a_coeffs"], 4)}')
print(f'  b_coeffs  : {np.round(params_temp["b_coeffs"], 4)}')
print(f'  sigma     : {params_temp["sigma"]:.6f}  yr^(-1/2)')
print(f'  eigenvalues (real): {np.round(params_temp["eigenvalues_real"], 4)}')
print(f'  loglik    : {params_temp["loglik"]:.3f}')
print(f'  AIC       : {params_temp["aic"]:.3f}')
print(f'  BIC       : {params_temp["bic"]:.3f}')
ev_real = params_temp['eigenvalues_real']
assert all(e < 0 for e in ev_real), 'STABILITY CHECK FAILED: positive eigenvalue!'
print('Stability check: PASSED (all eigenvalues have Re < 0)')
for ev in ev_real:
    tau_h = -1.0 / (ev * DT_YEARS)   # in hours
    print(f'  Re(lambda)={ev:.4f} yr^-1  =>  mean-reversion time = {-1/ev:.4f} yr = {tau_h:.1f} h')

## 2.  Log-price CARMA MLE

In [ ]:
PRICE_MLE_PATH = RES_DIR / 'carma_price_mle.json'

if PRICE_MLE_PATH.exists():
    print('Loading cached log-price MLE result …')
    params_price = load_params(PRICE_MLE_PATH)
else:
    print(f'Fitting CARMA({p_p},{q_p}) to log-price (multi-start MLE) …')
    _, params_price = fit_carma_mle(
        t_years     = t_price,
        y           = y_price,
        p           = p_p,
        q           = q_p,
        theta0_list = theta0_price,
        verbose     = True,
    )
    save_params(params_price, PRICE_MLE_PATH)

print(f'\nLog-price CARMA({p_p},{q_p}) MLE result:')
print(f'  a_coeffs  : {np.round(params_price["a_coeffs"], 4)}')
print(f'  b_coeffs  : {np.round(params_price["b_coeffs"], 4)}')
print(f'  sigma     : {params_price["sigma"]:.6f}  yr^(-1/2)')
print(f'  eigenvalues (real): {np.round(params_price["eigenvalues_real"], 4)}')
print(f'  loglik    : {params_price["loglik"]:.3f}')
print(f'  AIC       : {params_price["aic"]:.3f}')
print(f'  BIC       : {params_price["bic"]:.3f}')
ev_real_p = params_price['eigenvalues_real']
assert all(e < 0 for e in ev_real_p), 'STABILITY CHECK FAILED'
print('Stability check: PASSED')
for ev in ev_real_p:
    print(f'  Re(lambda)={ev:.4f} yr^-1  =>  τ = {-1/ev:.4f} yr = {-1/(ev*DT_YEARS):.1f} h')

## 3.  Run Kalman filter at MLE parameters and save filtered states

In [ ]:
# Temperature
res_temp = kalman_filter(
    t_years  = t_temp,
    y        = y_temp,
    a_coeffs = params_temp['a_coeffs'],
    b_coeffs = params_temp['b_coeffs'],
    sigma    = params_temp['sigma'],
)

np.save(RES_DIR / 'x_hat_temp.npy', res_temp['x_filt'])
np.save(RES_DIR / 'y_hat_temp.npy', res_temp['y_pred'])
print('Temperature Kalman filter:')
print(f'  loglik = {res_temp["loglik"]:.3f}')
rmse_t = np.sqrt(np.mean((y_temp - res_temp['y_pred'])**2))
print(f'  RMSE   = {rmse_t:.6f}  (log-temp units)')

In [ ]:
# Log-price
res_price = kalman_filter(
    t_years  = t_price,
    y        = y_price,
    a_coeffs = params_price['a_coeffs'],
    b_coeffs = params_price['b_coeffs'],
    sigma    = params_price['sigma'],
)

np.save(RES_DIR / 'x_hat_price.npy', res_price['x_filt'])
np.save(RES_DIR / 'y_hat_price.npy', res_price['y_pred'])
print('Log-price Kalman filter:')
print(f'  loglik = {res_price["loglik"]:.3f}')
rmse_p = np.sqrt(np.mean((y_price - res_price['y_pred'])**2))
print(f'  RMSE   = {rmse_p:.6f}  (log-EUR/MWh)')

## 4.  Residual diagnostics (standardised Kalman innovations)

In [ ]:
from scipy import stats as scipy_stats

for label, res, y_obs in [
        ('Temperature', res_temp, y_temp),
        ('Log-price',   res_price, y_price),
    ]:
    z = res['std_innov']
    lb = acorr_ljungbox(z, lags=[24, 48, 72], return_df=True)
    jb = scipy_stats.jarque_bera(z)
    print(f'\n{label} standardised innovations:')
    print(f'  mean={z.mean():.4f}  std={z.std():.4f}  skew={scipy_stats.skew(z):.3f}  kurt={scipy_stats.kurtosis(z, fisher=True):.3f}')
    print(f'  Ljung-Box p-values: lag24={lb["lb_pvalue"].iloc[0]:.4f}  lag48={lb["lb_pvalue"].iloc[1]:.4f}  lag72={lb["lb_pvalue"].iloc[2]:.4f}')
    print(f'  Jarque-Bera: stat={jb[0]:.1f}  p={jb[1]:.2e}')
    if lb['lb_pvalue'].iloc[0] > 0.05:
        print('  Whiteness check lag24: PASS')
    else:
        print('  Whiteness check lag24: FAIL – residual serial correlation')

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 8))

for row, (label, res, y_obs) in enumerate([
        ('Temperature', res_temp, y_temp),
        ('Log-price',   res_price, y_price),
    ]):
    z   = res['std_innov']
    idx = np.arange(len(y_obs))

    # Fitted vs observed (first 1000 points)
    ax = axes[row, 0]
    ax.plot(idx[:1000], y_obs[:1000], lw=0.8, alpha=0.7, label='observed')
    ax.plot(idx[:1000], res['y_pred'][:1000], lw=1.2, label='1-step pred')
    ax.set_title(f'{label}: observed vs 1-step prediction')
    ax.legend(fontsize=8)

    # ACF of standardised innovations
    ax = axes[row, 1]
    plot_acf(z, lags=72, ax=ax, markersize=2)
    ax.set_title(f'{label}: ACF of std innovations')

    # QQ plot vs N(0,1)
    ax = axes[row, 2]
    (osm, osr), (slope, intercept, _) = scipy_stats.probplot(z, dist='norm')
    ax.scatter(osm, osr, s=1, alpha=0.3)
    xl = np.array([osm.min(), osm.max()])
    ax.plot(xl, slope*xl + intercept, 'r-', lw=2)
    ax.set_xlabel('Theoretical N(0,1) quantiles')
    ax.set_ylabel('Sample quantiles')
    ax.set_title(f'{label}: QQ (std innovations vs Normal)')

plt.tight_layout()
plt.savefig(FIG_DIR / 'carma_mle_diagnostics.png', dpi=150)
plt.show()

## 5.  Model comparison: OU benchmark vs CARMA

In [ ]:
from carma_utils import neg_loglik_carma, arma_to_carma_init

comparison_rows = []

for label, y, t_yr, arma_key in [
        ('Temperature', y_temp, t_temp, 'temperature'),
        ('Log-price',   y_price, t_price, 'logprice'),
    ]:
    for p_c, q_c in [(1, 0), (2, 0), (2, 1), (3, 0), (3, 1)]:
        # Skip if more params than obs / 10
        if p_c + q_c + 1 > len(y) // 10:
            continue
        arma_pars = json.load(open(RES_DIR / 'arma_selected.json'))[arma_key]
        phi_init  = arma_pars['phi'][:p_c] if len(arma_pars['phi']) >= p_c else arma_pars['phi'] + [0.0]*(p_c-len(arma_pars['phi']))
        th0_list  = arma_to_carma_init(phi_init[:p_c], arma_pars['sigma2'], p_c, q_c, n_random=3)
        try:
            _, pars = fit_carma_mle(t_yr, y, p_c, q_c, th0_list, verbose=False)
            comparison_rows.append({'series': label, 'p': p_c, 'q': q_c,
                                    'loglik': pars['loglik'], 'AIC': pars['aic'], 'BIC': pars['bic']})
        except Exception as e:
            comparison_rows.append({'series': label, 'p': p_c, 'q': q_c,
                                    'loglik': np.nan, 'AIC': np.nan, 'BIC': np.nan})

df_compare = pd.DataFrame(comparison_rows)
print(df_compare.to_string(index=False, float_format='{:.1f}'.format))
df_compare.to_csv(RES_DIR / 'model_comparison.csv', index=False)
print(f'\nSaved → {RES_DIR}/model_comparison.csv')